# 05. SWE agent: от задачи до проверенного изменения

## Что агент пока не умеет

После `04` агент умеет делегировать анализ, но ещё не замыкает цикл изменения программного проекта.

> Coding agent полезен не тогда, когда умеет написать patch, а когда умеет понять задачу, локализовать изменение, применить его и проверить результат.


![SWE cycle](../visuals/openclaw_path/10_swe_cycle.svg)


## Подготовленный дефект

Не даём свободную задачу вроде «улучши архитектуру». Нужен маленький проверяемый кейс:

```text
VK connector должен разбивать сообщения длиннее 3500 символов на несколько частей.
Реализуй это и добавь тесты.
```

В проекте уже есть место, где bridge режет reply до `3500`. SWE-глава превращает это в явное поведение connector/test layer.


## Что показать в Studio

- план;
- вызов researcher;
- найденные файлы;
- `edit_file`;
- созданный тест;
- запуск `pytest`;
- вывод reviewer;
- итоговое описание diff.


In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

from deepagents import create_deep_agent
from dotenv import load_dotenv

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'pyproject.toml').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / '.env')

DEFAULT_MODEL = 'openrouter:tencent/hy3:free'

def model_name() -> str:
    return os.getenv('OPENCLAW_MODEL', DEFAULT_MODEL)

def write_text(path: str, content: str) -> Path:
    target = REPO_ROOT / path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content)
    return target

def write_json(path: str, payload: dict) -> Path:
    target = REPO_ROOT / path
    target.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + '\n')
    return target


In [ ]:
ENTRYPOINT = "\nfrom __future__ import annotations\n\nimport os\nfrom pathlib import Path\nfrom deepagents import create_deep_agent\nfrom dotenv import load_dotenv\nfrom connectors.jenkins import JENKINS_TOOLS\nfrom connectors.vk import VK_TOOLS\n\nload_dotenv()\nDEFAULT_MODEL = \"openrouter:tencent/hy3:free\"\n\n\ndef _backend():\n    root = Path(os.getenv(\"OPENCLAW_WORKSPACE\", \".\")).expanduser().resolve()\n    if os.getenv(\"OPENCLAW_ENABLE_LOCAL_SHELL\") == \"1\":\n        from deepagents.backends import LocalShellBackend\n        return LocalShellBackend(root_dir=root, virtual_mode=True, inherit_env=True, timeout=120, max_output_bytes=80_000)\n    from deepagents.backends import FilesystemBackend\n    return FilesystemBackend(root_dir=root, virtual_mode=True)\n\nTOOLS = [*JENKINS_TOOLS, *VK_TOOLS]\nSUBAGENTS = [\n    {\"name\": \"repo-researcher\", \"description\": \"Research repository facts before implementation.\", \"system_prompt\": \"Find relevant files, APIs, tests, and risks. Cite paths.\"},\n    {\"name\": \"reviewer\", \"description\": \"Review patches and test coverage.\", \"system_prompt\": \"Review correctness, regressions, tests, and security. Be specific.\"},\n]\nSWE_PROMPT = \"\"\"\\\nYou are OpenClaw SWE agent. Follow this issue-resolution loop:\n1. Reproduce or characterize the issue.\n2. Localize relevant files and tests.\n3. Patch the root cause.\n4. Add or update regression coverage.\n5. Run narrow tests first, then related checks.\n6. Ask reviewer to inspect the diff before final summary.\n\"\"\"\n\nswe_agent = create_deep_agent(\n    model=os.getenv(\"OPENCLAW_MODEL\", DEFAULT_MODEL),\n    tools=TOOLS,\n    system_prompt=SWE_PROMPT,\n    subagents=SUBAGENTS,\n    backend=_backend(),\n    interrupt_on={\"execute\": True, \"write_file\": True, \"edit_file\": True},\n)\n"
CONFIG = {"dependencies": ["."], "graphs": {"openclaw_05_swe": "./agents/openclaw_05_swe_agent.py:swe_agent"}, "env": ".env"}
print(write_text('agents/openclaw_05_swe_agent.py', ENTRYPOINT).relative_to(REPO_ROOT))
print(write_json('langgraph.openclaw_path.json', CONFIG).relative_to(REPO_ROOT))


## Проверка в LangGraph Studio

### Запрос

```text
VK connector должен разбивать сообщения длиннее 3500 символов на несколько частей. Реализуй это и добавь тесты.
```

### Ожидаемое поведение

1. Агент локализует `connectors/vk.py`, `scripts/vk_langgraph_bridge.py`, tests.
2. Создаёт/обновляет тест на длинное сообщение.
3. Вносит минимальный patch.
4. Запускает narrow pytest.
5. Просит reviewer проверить diff.

### Что изменилось относительно предыдущего этапа

Появился полный SWE workflow: issue → research → edit → test → review → summary.

### Текущее ограничение

Это учебный SWE контур. Нет production isolation, scheduler, полноценной memory subsystem и marketplace skills.
